# WorldPop API

**Endpoint:** `https://api.worldpop.org/v1/services/stats`
**API documentaqtion**: https://www.worldpop.org/sdi/advancedapi/

**Purpose:** Population density within 50km of the epicenter (data year 2020).  
Returns demographic statistics for a GeoJSON area.

**Used in:** `services/flink-enrichment/enrichment_job.py` → `fetch_worldpop()`

In [15]:
!pip install requests --quiet

In [16]:
import requests
import json
import math

In [25]:

BASE_URL = "https://api.worldpop.org/v1/services/stats"
EARTH_CIRCUMFERENCE_KM = 40_075

# Example coordinates: Naples, Italy
LAT = 40.85
LON = 14.27

In [46]:
def get_worldpop_params(lat: float, lon: float, radius_km: float) -> dict:
    lat_per_km = 360 / EARTH_CIRCUMFERENCE_KM
    lon_per_km = 360 / (EARTH_CIRCUMFERENCE_KM * math.cos(math.radians(lat)))

    coordinates = [[
        [
            lon + radius_km * math.sin(math.radians(60 * i)) * lon_per_km,
            lat + radius_km * math.cos(math.radians(60 * i)) * lat_per_km,
        ]
        for i in range(6)
    ]]

    geojson = {
        "type": "FeatureCollection",
        "features": 
        [
            {
                "type": "Feature",
                "properties": {},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": coordinates
                }
            }
        ]
    }

    return f"dataset=wpgppop&year=2020&geojson={json.dumps(geojson)}"

print(get_worldpop_params(LAT, LON, 100))

[[[14.27, 41.74831565814099], [15.298475583083933, 41.299157829070495], [15.298475583083933, 40.40084217092951], [14.27, 39.951684341859014], [13.241524416916066, 40.40084217092951], [13.241524416916066, 41.299157829070495]]]
dataset=wpgppop&year=2020&geojson={"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {}, "geometry": {"type": "Polygon", "coordinates": [[[14.27, 41.74831565814099], [15.298475583083933, 41.299157829070495], [15.298475583083933, 40.40084217092951], [14.27, 39.951684341859014], [13.241524416916066, 40.40084217092951], [13.241524416916066, 41.299157829070495]]]}}]}


In [52]:
def fetch_worldpop(lat: float, lon: float, radius_km: float) -> dict:
    """WorldPop: population density within 50km of epicenter (2020)."""
    try:
        resp = requests.get(
            "https://api.worldpop.org/v1/services/stats",
            params=get_worldpop_params(lat, lon, radius_km),
            timeout=100,
        )
        if resp.status_code == 200:
            task_id = resp.json().get("taskid", "")
            print(f"WorldPop task submitted, task ID: {task_id}")
            resp = requests.get(
                f"https://api.worldpop.org/v1/tasks/{task_id}",
                timeout=100,
            )
            return resp.json().get("data", {}).get("total_population", "")
    except Exception as e:
        print(f"WorldPop fetch failed: {e}", flush=True)
        return {}
    
population = fetch_worldpop(LAT, LON, 100.0)
print(population)

WorldPop task submitted, task ID: 00dc04a8-eef3-51eb-9d4f-d7d553225762
6089996.14


## Response structure

```json
{
  "status": "started",
  "status_code": 200,
  "error": false,
  "error_message": null,
  "data": {
    "total_population": 6089996.14
  },
  "taskid": "0ec0ad16-81bc-5332-97f1-ee295a439408",
  "startTime": "2026-05-12 19:36:15.435791",
  "endTime": "2026-05-12 19:36:19.011148",
  "executionTime": 4
}
```